# Dependence — compute (v3)

Reads the study's worlds and label stacks, writes **per-sensor parquet files**, interprets nothing. The
companion `dependence_analysis.ipynb` reads only these files (plus `storage_class_eval_v2`'s).

| file | one row per | content |
|---|---|---|
| `topology`, `nodes`, `node_district` | link / node / partition × node | `branch` / `source_bridge` / `loop`, orientation $\sigma$, downstream count; coordinates; district of each node |
| `branch` | world × branch link | exact series check, exact shares, **signed** coefficients $a^{\text{ex}}$, exact elasticities |
| `labels` | partition × link | **every column** of the label stack's mixture (home, tier, purity, $w_k$, `n_live`, `n_mass`, phases, `snr_home`, …), pumps flagged |
| `groups` | world × signal × basis × $\rho^\*$ | signal-group id on three bases: `raw`, `anom`, `resp` |
| `hub_elem`, `hub_link` | world × element (× link) | element's drift response; link $r^2$ / slope on it (v1, kept for comparison) |
| `hub_units`, `hub_unit_link` | world × unit (× link) | physical hub units; unit's drift response; link $r^2$ on the unit and its **unit storage term** |
| `switch_b`, `switch_a`, `switch_link` | world × link × pump / district | switch-event fit; channel split incl. own / cross demand and the residual demand channel |
| `composites` | world × composite gauge | known-truth mixtures built from exact branch flows, with every estimator's value |
| `profiles`, `series_week` | world × signal × window | weekly profiles (init, settled) and one hourly week per window, for links, hub elements, units, districts, composites |
| `placement`, `worlds` | world × placed sensor / world | **every column** of the world's `sensor_placement` (slot, `filled_as`, selection and label columns), pumps kept and flagged `is_pump`; windows, units, composite bases |
| `stack_gains`, `comp_switch` | partition × link × drift / world × composite × term | the label stack's own gains (`g`, `proj`, `nu`, `excite`, native, nulls); composite switch-fit coefficients |

**Changes from v2.** Full stack labels and gains, full placement rows (incl. pumps), composite switch
coefficients, and split-half nulls (`null_nat`, `null_shape`) for hub elements, hub units and composites — the
same definition as the links' null in v2, so the analysis can test a unit's drive and a composite's response
against noise.

**Changes from v1.** $a^{\text{ex}}$ carries the link orientation $\sigma$ (v1 compared signed fits with unsigned
truth). The demand channel is also written as a residual $g-g^{\text{sto}}$, and the switch demand term is split
into own and cross parts. The storage term is written per **hub unit** (each physical signal once). Signal groups
are also formed on de-seasoned anomalies and on drift-response profiles. New: series exports and composite gauges.

Run `storage_class_eval_v2` first: the analysis joins its `links` / `coefs` files.

In [1]:
%matplotlib inline
import warnings, logging, sys, pathlib, json, datetime
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse.csgraph import connected_components
from IPython.display import display, Markdown

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
assert REPO is not None, "run this from inside the fedwater repo"
LAB_DIR = REPO / "notebooks" / "fedwater_lab_refactor"     # adjust to your layout
for p in (REPO / "src", LAB_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from fedwater.placement import mixture_probe as mp, signal_probe as sp
from fedwater.reporting import Atlas

warnings.filterwarnings("ignore")
logging.getLogger("kedro").setLevel(logging.WARNING)

EXPERIMENTS = None              # None -> <project>/data/09_experiments
STUDY = None                    # None -> the latest study
ATLAS = Atlas(root=EXPERIMENTS)
STUDY = STUDY or ATLAS.studies()[1]
WORLDS = ATLAS.worlds(STUDY, ok_only=True).reset_index(drop=True)
OUT_DIR = REPO / "notebooks" / "outputs" / STUDY / "dependence"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RHOS = [0.80, 0.90, 0.95, 0.98, 0.995]    # signal-group thresholds; 0.80 / 0.90 are the relaxed ones
TAU = [0.5, 0.8, 0.95]          # composite gauges: volumetric share of the first district
RATIO = [0.5, 1.0, 2.0]         # composite gauges: injected storage sd / demand sd (init)
VERSION = 3
FLOW = "flow"
print(f"study {STUDY}: {len(WORLDS)} ok worlds  ->  {OUT_DIR}")

study ky72_districting_all_districts: 12 ok worlds  ->  c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\outputs\ky72_districting_all_districts\dependence


### Helpers

Same windows and conventions as `storage_class_eval_v2`: init = baseline window, settled = from the later of
the settle month and the drift's completion to the horizon; flows raw in `.inp` orientation; weekly profiles
$S$ per window; native elasticity $g^{\text{nat}}=\langle\Delta S,\Delta D_k\rangle/\lVert\Delta D_k\rVert^2$, shape
elasticity the same on $z$-scored profiles.

**Topology.** A link is a **bridge** if removing it disconnects the undirected graph (parallel links never cut).
A bridge is a `branch` if one side holds no tank or reservoir — its downstream set — and then, with
demand-driven hydraulics, $\hat Q_s(t)=\sigma_s\sum_{n\in\text{down}(s)}d_n(t)$ exactly. Sources on both sides:
`source_bridge`; anything else: `loop`.

| quantity | definition | why |
|---|---|---|
| anomaly | $x_t-\bar x_{h(t)}$, $\bar x_h$ = mean of the series at hour-of-week $h$ over the window | raw $\lvert\operatorname{corr}\rvert$ is dominated by the shared daily cycle and merges unrelated links |
| group basis `resp` | $\lvert\operatorname{corr}\rvert$ between drift-response profiles $\Delta S_s$ ($T$ points) | copies of one response are collinear here regardless of level |
| hub unit | connected component of the pump–tank graph whose edges are the `.inp` controls (pump $p$ switched on a condition of tank $t$) | a pump, the tank it fills and that tank's level are one physical signal |
| unit signal | number of the unit's pumps on; the unit's tank net inflow if no pump switches | one series per unit |

In [2]:
def target_probe(w):
    sch = w.catalog.load("gt_drift_schedule")
    if sch is None or not len(sch):
        return None
    tgt = str(sch.sort_values("drift_month")["district"].iloc[0])
    dy = w.catalog.load("districts")
    return sp.Probe(districts=dy["districts"], assets=dy.get("assets") or {},
                    demand=w.catalog.load("demand_series"), pressures=pd.DataFrame(),
                    flows=w.catalog.load("flows"), schedule=sch, time=w.params["time"],
                    params={"patterns": w.params["patterns"]}, drift={"tgt_district": tgt},
                    label=w.sim_hash)


def target_windows(P, settle, patterns):
    sm, n = P.steps_month, int(P.time["n_months"])
    if float((patterns or {}).get("seasonality_scale", 0.0)) > 0:
        return mp.season_windows(P, pad=int(settle["pad"]))
    final0 = int(P.phases()["final"][0])
    try:
        s_lo, _ = mp.settled_window(P, ref_months=int(settle["ref_months"]),
                                    slack=float(settle["slack"]), pad=int(settle["pad"]),
                                    min_months=0)
        start = max(s_lo // sm, final0)
    except ValueError:
        start = final0
    need = int(settle.get("min_months", 3))
    if n - start < need:
        raise ValueError(f"unsettled: {n - start} settled month(s), {need} needed")
    return mp.baseline_window(P), (start * sm, n * sm)


def storage_map(wn, cols):
    cols = set(cols)
    pumps = [p for p in wn.pump_name_list if p in cols]
    tanks = {}
    for t in wn.tank_name_list:
        inc = [(ln, 1.0 if wn.get_link(ln).end_node_name == t else -1.0)
               for ln in wn.get_links_for_node(t) if ln in cols]
        if inc:
            tanks[t] = inc
    return pumps, tanks


def hub_units(wn, pumps, tanks):
    # physical hub units: connected components of the pump-tank graph built from the .inp controls
    # (a control that switches pump p on a condition on tank t links p and t); unlinked
    # pumps / tanks are units of their own
    G = nx.Graph()
    G.add_nodes_from([("pump", p) for p in pumps] + [("tank", t) for t in tanks])
    for _, ctl in wn.controls():
        req = {getattr(o, "name", None) for o in ctl.requires()}
        ps, ts = [p for p in pumps if p in req], [t for t in tanks if t in req]
        for p in ps:
            for t in ts:
                G.add_edge(("pump", p), ("tank", t))
    units = {}
    for comp in nx.connected_components(G):
        ts = sorted(n for k, n in comp if k == "tank")
        ps = sorted(n for k, n in comp if k == "pump")
        units["unit:" + "+".join(ts or ps)] = {"tanks": ts, "pumps": ps}
    return units


def pump_on(q):
    q = np.asarray(q, float)
    return np.abs(q) > 1e-6 * max(np.abs(q).max(), 1e-12)


def ols(X, Y):
    B, *_ = np.linalg.lstsq(X, Y, rcond=None)
    return B


def r2(Y, Yhat):
    tot = ((Y - Y.mean(axis=0)) ** 2).sum(axis=0)
    res = ((Y - Yhat) ** 2).sum(axis=0)
    return np.where(tot > 1e-12, 1.0 - res / np.where(tot > 1e-12, tot, 1.0), np.nan)


def zrows(A):
    s = A.std(axis=1, keepdims=True)
    return (A - A.mean(axis=1, keepdims=True)) / np.where(s > 1e-12, s, np.nan)


def prof(Xw, sd, lo):
    Xw = np.asarray(Xw, float)
    Xw = Xw if Xw.ndim > 1 else Xw[:, None]
    return np.atleast_2d(sp.weekly_profile(Xw, sd, step0=lo))           # (m, T)


def pair(X, sd, wi, wf):
    # weekly profiles of every column of X on the init and settled windows: (m, T) each
    (i0, i1), (f0, f1) = wi, wf
    return prof(X[i0:i1], sd, i0), prof(X[f0:f1], sd, f0)


def gains(pI, pF, dI, dF):
    # native and shape elasticity of every row of (pI, pF) to the district profiles (dI, dF)
    dD = dF - dI
    g_nat = ((pF - pI) @ dD.T)[:, 0] / float((dD ** 2).sum())
    dz = zrows(dF) - zrows(dI)
    g_shape = ((zrows(pF) - zrows(pI)) @ dz.T)[:, 0] / float((dz ** 2).sum())
    return g_nat, g_shape


def split_null(X, sd, wi, wf, dI, dF):
    # split-half null of the native and shape gains (the stack's / v2's definition): profile change between
    # the two halves of the init window, rescaled to the init-vs-settled sample sizes, projected like the gains
    (i0, i1), (f0, f1) = wi, wf
    mid = i0 + ((i1 - i0) // (2 * sd)) * sd
    A, B = prof(X[i0:mid], sd, i0), prof(X[mid:i1], sd, mid)
    n_h, n_a, n_b = (mid - i0) / sd, (i1 - i0) / sd, (f1 - f0) / sd
    s = float(np.sqrt((1 / n_a + 1 / n_b) / (2 / n_h)))
    dD = dF - dI
    dz = zrows(dF) - zrows(dI)
    n_nat = ((B - A) @ dD.T)[:, 0] * s / float((dD ** 2).sum())
    n_shape = ((zrows(B) - zrows(A)) @ dz.T)[:, 0] * s / float((dz ** 2).sum())
    return n_nat, n_shape


def elasticities(X, D_k, sd, wi, wf):
    return gains(*pair(X, sd, wi, wf), *pair(D_k, sd, wi, wf))


def deseason(X, sd, lo):
    # anomaly: each series minus its own hour-of-week mean over the window
    X = np.asarray(X, float)
    P_ = 7 * sd
    idx = (np.arange(len(X)) + lo) % P_
    M = np.zeros((P_, X.shape[1]))
    for t in range(P_):
        M[t] = X[idx == t].mean(axis=0)
    return X - M[idx]


def signal_groups(Y, rhos):
    # connected components of |corr| >= rho over the columns of Y; constant columns stay singletons
    sdv = Y.std(axis=0)
    dead = sdv <= 1e-12
    Z = np.nan_to_num((Y - Y.mean(axis=0)) / np.where(dead, np.nan, sdv))
    C = np.abs(Z.T @ Z / len(Z))
    out = {}
    for rho in rhos:
        A = (C >= rho).astype(np.int8)
        A[dead, :] = 0
        A[:, dead] = 0
        np.fill_diagonal(A, 1)
        out[rho] = connected_components(A, directed=False)[1]
    return out


def switch_fit(Y, ONi, Di):
    # first-difference regression on the init window: dY = c + b dU + a dD + e (columns of Y independent)
    dY, dU, dD = np.diff(Y, axis=0), np.diff(ONi, axis=0), np.diff(Di, axis=0)
    X = np.column_stack([np.ones(len(dY)), dU, dD])
    B = ols(X, dY)
    res = dY - X @ B
    s2 = (res ** 2).sum(axis=0) / max(len(dY) - X.shape[1], 1)
    se = np.sqrt(np.outer(np.diag(np.linalg.pinv(X.T @ X)), s2))
    t = B / np.where(se > 0, se, np.nan)
    nb = dU.shape[1]
    return {"b": B[1:1 + nb], "a": B[1 + nb:], "tb": t[1:1 + nb], "ta": t[1 + nb:],
            "r2": r2(dY, X @ B), "events": np.abs(dU).sum(axis=0)}


def wide(kind, ids, A, window, prefix="h"):
    A = np.asarray(A, np.float32)
    df = pd.DataFrame(A, columns=[f"{prefix}{t:03d}" for t in range(A.shape[1])])
    df.insert(0, "window", window)
    df.insert(0, "signal", list(ids))
    df.insert(0, "kind", kind)
    return df


def topology(wn):
    G = nx.Graph()
    pairs = Counter()
    ends = {}
    for ln in wn.link_name_list:
        l = wn.get_link(ln)
        u, v = l.start_node_name, l.end_node_name
        ends[ln] = (u, v)
        G.add_edge(u, v)
        pairs[frozenset((u, v))] += 1
    sources = set(wn.tank_name_list) | set(wn.reservoir_name_list)
    bridges = {frozenset(e) for e in nx.bridges(G)}
    rows, down = [], {}
    for ln, (u, v) in ends.items():
        e = frozenset((u, v))
        base = {"element": ln, "start": u, "end": v, "link_type": type(wn.get_link(ln)).__name__}
        if e not in bridges or pairs[e] > 1:
            rows.append({**base, "topo": "loop", "sigma": np.nan, "n_down": np.nan}); continue
        G.remove_edge(u, v)
        cu = nx.node_connected_component(G, u)
        G.add_edge(u, v)
        cv = set(G.nodes) - cu
        su, sv = bool(cu & sources), bool(cv & sources)
        if su and sv:
            rows.append({**base, "topo": "source_bridge", "sigma": np.nan, "n_down": np.nan})
        elif su or sv:
            dn = cv if su else cu
            down[ln] = dn
            rows.append({**base, "topo": "branch", "sigma": 1.0 if v in dn else -1.0,
                         "n_down": len(dn)})
        else:
            rows.append({**base, "topo": "isolated", "sigma": np.nan, "n_down": np.nan})
    return pd.DataFrame(rows), down


## The pass

Per world: **(B)** branch truth; **(C)** signal groups; **(D)** hub elements and units; **(E)** placed sensors;
**(F)** composite gauges; **(G)** switch-event split for links and composites; **(H)** series exports.

**(G) switch events**, init window, first differences of hourly series,
$\Delta q_{s,t}=c_s+\sum_p b_{s,p}\Delta u_{p,t}+\sum_d a_{s,d}\Delta D_{d,t}+\varepsilon_{s,t}$, $u_p\in\{0,1\}$. With
$\gamma_{p,k}=\langle\Delta U_p,\Delta D_k\rangle/\lVert\Delta D_k\rVert^2$ and $\pi_{d,k}=\langle\Delta D_d,\Delta D_k\rangle/\lVert\Delta D_k\rVert^2$:

| term | definition |
|---|---|
| storage, per unit | $g^{\text{sto}}_{s,u}=\sum_{p\in u}b_{s,p}\gamma_{p,k}$; $\sum_u g^{\text{sto}}_{s,u}=g^{\text{sto}}_s$ |
| demand, own / cross | $a_{s,k}$ / $\sum_{d\ne k}a_{s,d}\pi_{d,k}$ |
| demand, residual | $g^{\text{dem,resid}}_s=g^{\text{nat}}_s-g^{\text{sto}}_s$ (tank-level effects land here) |

Native only: the shape transform is not linear, so no shape split exists.

**(F) composite gauges** — mixtures with known truth, built from simulated flows of pure branch links. Per
district $d$, $\text{base}_d=\sigma_bQ_b/m_b$, $b$ = the pure branch of $d$ with the largest oriented init mean flow
$m_b$ (init mean 1; init windows are shared within a partition, so all its worlds use the same bases).

| kind | gauge | truth |
|---|---|---|
| `single` | $\text{base}_d$ | share 1, $a_d=a^{\text{ex}}_b/m_b$ |
| `pair` | $\tau\,\text{base}_j+(1-\tau)\,\text{base}_k$, $\tau\in\{0.5,0.8,0.95\}$ | init volumetric share $\tau$; $a_j=\tau a^{\text{ex}}_{b_j}/m_{b_j}$ |
| `storage` | $\text{base}_d+\lambda H_u$, $H_u=\sum_{p\in u}\lvert q_p\rvert$, $\lambda=r\,\operatorname{sd}(\text{base}_d)/\operatorname{sd}(H_u)$, $r\in\{0.5,1,2\}$ | $g^{\text{dem}}=g(\text{base}_d)$, $g^{\text{sto}}=\lambda\,g(H_u)$ |

Tests: mixture weights against the volumetric share (gains over the $K$ worlds of a partition), shape vs native,
recovery of an injected storage share by the switch split, and coefficient leakage onto districts not in the
gauge. A composite is not a pipe: it tests the **estimators**, not the hydraulics. Pump flow is not exactly
$b\,u$ (it depends on head) — that gap is part of the test.

In [3]:
OUT = {k: [] for k in ("topology", "nodes", "node_district", "branch", "labels", "groups", "hub_elem",
                       "hub_link", "hub_units", "hub_unit_link", "switch_b", "switch_a", "switch_link",
                       "composites", "comp_switch", "profiles", "series_week", "placement",
                       "stack_gains", "worlds")}
TOPO, DOWN, LAB_DONE, issues = {}, {}, set(), []

for wi, r in WORLDS.iterrows():
    h = r["sim_hash"]
    try:
        w = ATLAS.world(h)
        L = w.label()
        P = target_probe(w)
        if L is None or P is None:
            raise ValueError("no label stack or no drift")
        WI, WF = target_windows(P, w.params["sensor_placement"]["probe"]["settle"], w.params["patterns"])
        (i0, i1), (f0, f1) = WI, WF
        sd, sm = P.steps_day, P.steps_month
        TW = 7 * sd                                                   # one week of steps
        net, meth = r["network"], r["districting"]
        wn = w.network_model
        if net not in TOPO:
            TOPO[net], DOWN[net] = topology(wn)
            OUT["topology"].append(TOPO[net].assign(network=net))
            OUT["nodes"].append(pd.DataFrame(
                [{"network": net, "node": n, "x": wn.get_node(n).coordinates[0],
                  "y": wn.get_node(n).coordinates[1],
                  "kind": ("tank" if n in wn.tank_name_list else
                           "reservoir" if n in wn.reservoir_name_list else "junction"),
                  "district": None} for n in wn.node_name_list]))
        topo = TOPO[net].set_index("element")
        names = P.names
        drift = P.drift["tgt_district"]
        kd = names.index(drift)
        meta = {"network": net, "districting": meth, "world": h[:8], "drift": drift}
        M = L.mixture[L.mixture["kind"] == FLOW].copy()
        M = M[~M["element"].astype(str).isin(set(wn.pump_name_list))]
        M = M[M["element"].astype(str).isin(P.flows.columns)].reset_index(drop=True)
        els = M["element"].astype(str).tolist()
        if (net, meth) not in LAB_DONE:
            Mall = L.mixture[L.mixture["kind"] == FLOW].copy()
            Mall["is_pump"] = Mall["element"].astype(str).isin(set(wn.pump_name_list))
            OUT["labels"].append(Mall.assign(network=net, districting=meth))       # every column of the stack
            G_ = getattr(L, "gains", None)
            if G_ is not None and len(G_):
                G_ = G_[G_["kind"] == FLOW].copy()
                G_["is_pump"] = G_["element"].astype(str).isin(set(wn.pump_name_list))
                OUT["stack_gains"].append(G_.assign(network=net, districting=meth))
            OUT["node_district"].append(pd.DataFrame(
                [{"network": net, "districting": meth, "node": str(n), "district": d}
                 for d, nodes in P.districts.items() for n in nodes]))
            LAB_DONE.add((net, meth))
        Q = P.flows[els].to_numpy(float)
        D = P.district_demand[names].to_numpy(float)
        node2d = {str(n): d for d, nodes in P.districts.items() for n in nodes}
        dI, dF = pair(D[:, kd], sd, WI, WF)                            # drifting district's profiles
        dDn = dF - dI
        nD = float((dDn ** 2).sum())
        proj = lambda dX: (dX @ dDn.T)[:, 0] / nD
        qI, qF = pair(Q, sd, WI, WF)
        DI, DF = pair(D, sd, WI, WF)
        g_link_nat, g_link_shape = gains(qI, qF, dI, dF)
        T_ = qI.shape[1]                                               # profile length (one week)
        pD = proj(DF - DI)                                             # projection of every district's change on k's

        # ---- (B) branch truth -----------------------------------------------------
        dem = P.demand
        dcols = [c for c in dem.columns if c in set(wn.junction_name_list)]
        col_ix = {c: i for i, c in enumerate(dcols)}
        DM = dem[dcols].to_numpy(float)
        br = [e for e in els if topo.at[e, "topo"] == "branch"]
        ids = M.set_index(M["element"].astype(str))["id"]
        base_comp = {}                                                 # district -> (series, element, coefficient)
        if br:
            Bmat = np.zeros((len(br), len(dcols)))
            Bk = {d: np.zeros((len(br), len(dcols))) for d in names}
            for b_, e in enumerate(br):
                for n in DOWN[net][e]:
                    if n in col_ix:
                        Bmat[b_, col_ix[n]] = 1.0
                        d = node2d.get(n)
                        if d in Bk:
                            Bk[d][b_, col_ix[n]] = 1.0
            sig = topo.loc[br, "sigma"].to_numpy(float)
            Qhat = (DM @ Bmat.T) * sig                                  # (n, n_branch)
            Qsim = P.flows[br].to_numpy(float)
            scale = np.maximum(np.abs(Qsim).mean(axis=0), 1e-12)
            err_max = np.abs(Qsim - Qhat).max(axis=0) / scale
            err_rms = np.sqrt(((Qsim - Qhat) ** 2).mean(axis=0)) / scale
            dbar_i, dbar_f = DM[i0:i1].mean(axis=0), DM[f0:f1].mean(axis=0)
            Dtot_i = {d: sum(dbar_i[col_ix[n]] for n in P.districts[d] if n in col_ix) for d in names}
            tot_i, tot_f = Bmat @ dbar_i, Bmat @ dbar_f
            gx_nat, gx_shape = elasticities(Qhat, D[:, kd], sd, WI, WF)
            gx_nat_k, _ = elasticities((DM @ Bk[drift].T) * sig, D[:, kd], sd, WI, WF)
            g_sim_nat, g_sim_shape = elasticities(Qsim, D[:, kd], sd, WI, WF)
            share_i = {d: (Bk[d] @ dbar_i) / np.where(tot_i > 0, tot_i, np.nan) for d in names}
            m_b = (sig[:, None] * Qsim[i0:i1].T).mean(axis=1)                    # oriented init mean flow
            for b_, e in enumerate(br):
                row = {**meta, "id": ids[e], "element": e, "sigma": sig[b_], "n_down": int(Bmat[b_].sum()),
                       "series_err_max": err_max[b_], "series_err_rms": err_rms[b_],
                       "g_exact_nat": gx_nat[b_], "g_exact_nat_k": gx_nat_k[b_],
                       "g_exact_shape": gx_shape[b_], "g_sim_nat": g_sim_nat[b_],
                       "g_sim_shape": g_sim_shape[b_]}
                for d in names:
                    vi, vf = Bk[d][b_] @ dbar_i, Bk[d][b_] @ dbar_f
                    row[f"w_exact_init_{d}"] = vi / tot_i[b_] if tot_i[b_] > 0 else np.nan
                    row[f"w_exact_final_{d}"] = vf / tot_f[b_] if tot_f[b_] > 0 else np.nan
                    a_un = vi / Dtot_i[d] if Dtot_i[d] > 0 else np.nan
                    row[f"a_exact_{d}"] = sig[b_] * a_un                 # signed: same orientation as the flow
                    row[f"a_exact_unsigned_{d}"] = a_un
                OUT["branch"].append(row)
            # composite bases: per district the pure branch with the largest init flow, oriented and
            # normalised to init mean 1 (init window is shared by the worlds of a partition -> same choice)
            for d in names:
                cand = [(m_b[b_], b_) for b_ in range(len(br)) if share_i[d][b_] >= 0.999 and m_b[b_] > 0]
                if cand:
                    mb, b_ = max(cand)
                    a_un = (Bk[d][b_] @ dbar_i) / Dtot_i[d] if Dtot_i[d] > 0 else np.nan
                    base_comp[d] = (sig[b_] * Qsim[:, b_] / mb, br[b_], a_un / mb)
        del DM

        # ---- hub signals (as v2) and hub units -------------------------------------
        pumps, tanks = storage_map(wn, P.flows.columns)
        pres = w.catalog.load("pressures")
        LEV = {t: pres[t].to_numpy(float) for t in tanks if t in pres.columns}
        del pres
        H_cols, H, ON = [], [], {}
        for p in pumps:
            ON[p] = pump_on(P.flows[p].to_numpy(float))
            if ON[p].std() > 0:
                H_cols.append(f"pump:{p}"); H.append(ON[p].astype(float))
        for t in tanks:
            if t in LEV and LEV[t].std() > 0:
                H_cols.append(f"level:{t}"); H.append(LEV[t])
        TF = {t: np.sum([s * P.flows[ln].to_numpy(float) for ln, s in inc], axis=0) for t, inc in tanks.items()}
        for t in tanks:
            if TF[t].std() > 0:
                H_cols.append(f"tankflow:{t}"); H.append(TF[t])
        H = np.column_stack(H) if H else np.zeros((len(Q), 0))
        act = [p for p in pumps if ON[p][i0:i1].std() > 0]               # pumps that switch in init
        UNITS = hub_units(wn, pumps, list(tanks))
        U_cols, U_sig, U_kind = [], [], []
        for u, mem in UNITS.items():
            live = [p for p in mem["pumps"] if ON[p].std() > 0]
            if live:
                U_cols.append(u); U_kind.append("pumps_on"); U_sig.append(np.sum([ON[p] for p in live], axis=0).astype(float))
            elif mem["tanks"] and np.std(np.sum([TF[t] for t in mem["tanks"]], axis=0)) > 0:
                U_cols.append(u); U_kind.append("tankflow"); U_sig.append(np.sum([TF[t] for t in mem["tanks"]], axis=0))
        U_sig = np.column_stack(U_sig) if U_sig else np.zeros((len(Q), 0))

        # ---- (C) signal groups on three bases --------------------------------------
        sig_ids = list(M["id"]) + H_cols
        kinds = ["link"] * len(els) + ["hub"] * len(H_cols)
        hI, hF = pair(H, sd, WI, WF) if H.shape[1] else (np.zeros((0, T_)), np.zeros((0, T_)))
        Ys = np.column_stack([Q[f0:f1], H[f0:f1]])
        bases = {"raw": Ys, "anom": deseason(Ys, sd, f0),
                 "resp": np.vstack([qF - qI, hF - hI]).T}               # (T, m): drift-response profiles
        for basis, Yb in bases.items():
            for rho, lab in signal_groups(Yb, RHOS).items():
                OUT["groups"].append(pd.DataFrame({"signal": sig_ids, "kind": kinds, "basis": basis,
                                                   "rho": rho, "group": lab}).assign(**meta))
        del bases, Ys

        # ---- (D) hub elements and units: own response; links' r^2 (settled) and slope (init) ----
        if H.shape[1]:
            hg_nat, hg_shape = gains(hI, hF, dI, dF)
            hn_nat, hn_shape = split_null(H, sd, WI, WF, dI, dF)
            for k_, c in enumerate(H_cols):
                OUT["hub_elem"].append({**meta, "element": c, "g_nat": hg_nat[k_], "g_shape": hg_shape[k_],
                                        "null_nat": hn_nat[k_], "null_shape": hn_shape[k_],
                                        "switches": int(np.abs(np.diff(ON[c.split(':', 1)[1]].astype(int))).sum())
                                        if c.startswith("pump:") else np.nan})
            Hs, Qs = H[f0:f1], Q[f0:f1]
            Hz = (Hs - Hs.mean(0)) / np.where(Hs.std(0) > 1e-12, Hs.std(0), np.nan)
            Qz = (Qs - Qs.mean(0)) / np.where(Qs.std(0) > 1e-12, Qs.std(0), np.nan)
            R2h = (Qz.T @ Hz / len(Hz)) ** 2
            Hi, Qi = H[i0:i1], Q[i0:i1]
            beta = ((Qi - Qi.mean(0)).T @ (Hi - Hi.mean(0)) / len(Hi)) / \
                np.where(Hi.var(0) > 1e-12, Hi.var(0), np.nan)
            OUT["hub_link"].append(pd.DataFrame({
                "id": np.repeat(M["id"].to_numpy(), len(H_cols)),
                "element": np.tile(H_cols, len(els)),
                "r2": R2h.ravel(), "beta": beta.ravel()}).assign(**meta))
        uI, uF = pair(U_sig, sd, WI, WF) if U_sig.shape[1] else (np.zeros((0, T_)), np.zeros((0, T_)))
        if U_sig.shape[1]:
            ug_nat, ug_shape = gains(uI, uF, dI, dF)
            un_nat, un_shape = split_null(U_sig, sd, WI, WF, dI, dF)
            Us, Qs = U_sig[f0:f1], Q[f0:f1]
            Uz = (Us - Us.mean(0)) / np.where(Us.std(0) > 1e-12, Us.std(0), np.nan)
            Qz = (Qs - Qs.mean(0)) / np.where(Qs.std(0) > 1e-12, Qs.std(0), np.nan)
            R2u = (Qz.T @ Uz / len(Uz)) ** 2                            # (links, units)
            for k_, u in enumerate(U_cols):
                OUT["hub_units"].append({**meta, "unit": u, "tanks": ",".join(UNITS[u]["tanks"]),
                                         "pumps": ",".join(UNITS[u]["pumps"]),
                                         "active_pumps": ",".join(p for p in UNITS[u]["pumps"] if p in act),
                                         "signal": U_kind[k_], "g_nat": ug_nat[k_], "g_shape": ug_shape[k_],
                                         "null_nat": un_nat[k_], "null_shape": un_shape[k_]})

        # ---- (E) placed sensors ---------------------------------------------------------
        PL = w.catalog.load("sensor_placement")
        PL = PL[PL["kind"] == FLOW].copy()
        PL["is_pump"] = PL["element"].astype(str).isin(set(wn.pump_name_list))
        PL = PL.drop(columns=[c for c in ("network", "districting") if c in PL.columns])
        OUT["placement"].append(PL.rename(columns={"sensor": "id"}).assign(**meta))   # every column; pumps flagged

        # ---- (F) composite gauges: known mixtures built from exact branch flows -------------
        C_cols, C_meta, C_ser = [], [], []
        def add_comp(series, spec):
            C_cols.append(spec["comp_id"]); C_meta.append(spec); C_ser.append(series)
        for d, (s, e, a_) in base_comp.items():
            add_comp(s, {"comp_id": f"single:{d}", "kind": "single", "parts": d, "tau": 1.0,
                         "unit": None, "ratio": 0.0, "lam": 0.0, "a_true": {d: a_}})
        bd = list(base_comp)
        for x_, d1 in enumerate(bd):
            for d2 in bd[x_ + 1:]:
                for tau in TAU:
                    s1, _, a1 = base_comp[d1]
                    s2, _, a2 = base_comp[d2]
                    add_comp(tau * s1 + (1 - tau) * s2,
                             {"comp_id": f"pair:{d1}|{d2}:{tau}", "kind": "pair", "parts": f"{d1}|{d2}",
                              "tau": tau, "unit": None, "ratio": 0.0, "lam": 0.0,
                              "a_true": {d1: tau * a1, d2: (1 - tau) * a2}})
        STO = {}                                                          # unit -> summed flow of its live pumps
        for u in U_cols:
            live = [p for p in UNITS[u]["pumps"] if p in act]
            if live:
                STO[u] = np.sum([np.abs(P.flows[p].to_numpy(float)) for p in live], axis=0)
        for d, (s, e, a_) in base_comp.items():
            for u, hs in STO.items():
                for ratio in RATIO:
                    lam = ratio * s[i0:i1].std() / max(hs[i0:i1].std(), 1e-12)
                    add_comp(s + lam * hs, {"comp_id": f"sto:{d}:{u}:{ratio}", "kind": "storage", "parts": d,
                                            "tau": 1.0, "unit": u, "ratio": ratio, "lam": lam,
                                            "a_true": {d: a_}})
        Cm = np.column_stack(C_ser) if C_ser else np.zeros((len(Q), 0))
        cI, cF = pair(Cm, sd, WI, WF) if Cm.shape[1] else (np.zeros((0, T_)), np.zeros((0, T_)))

        # ---- (G) switch events on the init window: links and composites --------------------
        ONi = np.column_stack([ON[p][i0:i1].astype(float) for p in act]) if act else None
        if act:
            UP = {p: pair(ON[p].astype(float), sd, WI, WF) for p in act}
            dU_prof = np.vstack([UP[p][1] - UP[p][0] for p in act])    # (nb, T)
            gam = proj(dU_prof)                                         # each pump's on-state change on k's
            SW = switch_fit(Q[i0:i1], ONi, D[i0:i1])
            bs, as_ = SW["b"], SW["a"]
            for k_, p in enumerate(act):
                OUT["switch_b"].append(pd.DataFrame({"id": M["id"], "pump": p, "b": bs[k_], "t": SW["tb"][k_],
                                                     "events": int(SW["events"][k_])}).assign(**meta))
            for k_, d in enumerate(names):
                OUT["switch_a"].append(pd.DataFrame({"id": M["id"], "district": d, "a": as_[k_],
                                                     "t": SW["ta"][k_]}).assign(**meta))
            g_sto = bs.T @ gam
            g_dem = as_.T @ pD
            g_own = as_[kd]
            OUT["switch_link"].append(pd.DataFrame({
                "id": M["id"], "element": els, "R2_fd": SW["r2"], "g_nat": g_link_nat,
                "g_sto_switch": g_sto, "g_dem_switch": g_dem, "g_dem_own_switch": g_own,
                "g_dem_cross_switch": g_dem - g_own, "g_dem_resid": g_link_nat - g_sto,
                "g_rest_switch": g_link_nat - g_sto - g_dem}).assign(**meta))
            ix = {p: k_ for k_, p in enumerate(act)}
            for k_, u in enumerate(U_cols):
                up = [ix[p] for p in UNITS[u]["pumps"] if p in ix]
                OUT["hub_unit_link"].append(pd.DataFrame({
                    "id": M["id"], "unit": u, "r2": R2u[:, k_],
                    "g_sto_unit": bs[up].T @ gam[up] if up else np.nan}).assign(**meta))
        elif U_cols:
            for k_, u in enumerate(U_cols):
                OUT["hub_unit_link"].append(pd.DataFrame({"id": M["id"], "unit": u, "r2": R2u[:, k_],
                                                          "g_sto_unit": np.nan}).assign(**meta))
        if C_cols:
            c_nat, c_shape = gains(cI, cF, dI, dF)
            cn_nat, cn_shape = split_null(Cm, sd, WI, WF, dI, dF)
            rows_c = []
            if act:
                SC = switch_fit(Cm[i0:i1], ONi, D[i0:i1])
                c_sto, c_dem = SC["b"].T @ gam, SC["a"].T @ pD
                for k_, p in enumerate(act):
                    OUT["comp_switch"].append(pd.DataFrame({"comp_id": C_cols, "term": p, "term_kind": "pump",
                                                            "coef": SC["b"][k_], "t": SC["tb"][k_]}).assign(**meta))
                for k_, d in enumerate(names):
                    OUT["comp_switch"].append(pd.DataFrame({"comp_id": C_cols, "term": d, "term_kind": "district",
                                                            "coef": SC["a"][k_], "t": SC["ta"][k_]}).assign(**meta))
            for j_, spec in enumerate(C_meta):
                parts = spec["parts"].split("|")
                comp_dem = sum((spec["tau"] if q_ == 0 else 1 - spec["tau"]) * base_comp[d][0]
                               for q_, d in enumerate(parts))
                tI, tF = pair(comp_dem, sd, WI, WF)
                g_dem_true = float(proj(tF - tI)[0])
                g_sto_true = 0.0
                if spec["kind"] == "storage":
                    sI, sF = pair(STO[spec["unit"]], sd, WI, WF)
                    g_sto_true = float(spec["lam"] * proj(sF - sI)[0])
                row = {**meta, **{k: v for k, v in spec.items() if k != "a_true"},
                       "g_dem_true": g_dem_true, "g_sto_true": g_sto_true,
                       "g_nat": c_nat[j_], "g_shape": c_shape[j_],
                       "null_nat": cn_nat[j_], "null_shape": cn_shape[j_]}
                for q_, d in enumerate(names):
                    row[f"a_true_{d}"] = spec["a_true"].get(d, 0.0)
                    wI = (spec["tau"] if d == parts[0] else (1 - spec["tau"]) if d in parts else 0.0)
                    row[f"share_init_{d}"] = wI
                for q_, d in enumerate(parts):
                    wt = spec["tau"] if q_ == 0 else 1 - spec["tau"]
                    row[f"vol_final_{d}"] = wt * base_comp[d][0][f0:f1].mean()
                if act:
                    row.update({"R2_fd": SC["r2"][j_], "g_sto_switch": c_sto[j_], "g_dem_switch": c_dem[j_],
                                "g_dem_own_switch": SC["a"][kd, j_], "g_dem_resid": c_nat[j_] - c_sto[j_]})
                    for q_, d in enumerate(names):
                        row[f"a_{d}"] = SC["a"][q_, j_]
                rows_c.append(row)
            OUT["composites"].append(pd.DataFrame(rows_c))

        # ---- (H) series exports: weekly profiles (init, settled) and one hourly week each ----
        groups_ = (("link", list(M["id"]), Q, qI, qF), ("hub", H_cols, H, hI, hF),
                   ("unit", U_cols, U_sig, uI, uF), ("district", names, D, DI, DF),
                   ("composite", C_cols, Cm, cI, cF))
        for kind, sid, X, xI, xF in groups_:
            if not len(sid):
                continue
            OUT["profiles"].append(pd.concat([wide(kind, sid, xI, "init"), wide(kind, sid, xF, "settled")])
                                   .assign(**meta))
            OUT["series_week"].append(pd.concat([wide(kind, sid, X[i1 - TW:i1].T, "init", "t"),
                                                 wide(kind, sid, X[f1 - TW:f1].T, "settled", "t")])
                                      .assign(**meta))

        OUT["worlds"].append({**meta, "i0": i0, "i1": i1, "f0": f0, "f1": f1, "steps_day": sd,
                              "week_init_start": i1 - TW, "week_settled_start": f1 - TW,
                              "n_links": len(els), "n_branch": len(br),
                              "branch_err_max_median": float(np.median(err_max)) if br else np.nan,
                              "branch_err_max_max": float(np.max(err_max)) if br else np.nan,
                              "hub_elements": ",".join(H_cols), "switch_pumps": ",".join(act),
                              "units": ";".join(f"{u}[{U_kind[k_]}]" for k_, u in enumerate(U_cols)),
                              "composite_bases": ",".join(f"{d}:{v[1]}" for d, v in base_comp.items()),
                              "n_composites": len(C_cols)})
        print(f"\r  {wi + 1}/{len(WORLDS)} — {meth} {drift}            ", end="", flush=True)
        del P, Q, D, H, Cm
    except Exception as exc:                              # reported, never swallowed silently
        issues.append({"world": h[:8], "districting": r["districting"],
                       "drift": r["drift_district"], "issue": f"{type(exc).__name__}: {exc}"[:240]})
print()
if issues:
    display(Markdown("**Worlds not analysed**")); display(pd.DataFrame(issues))
if not OUT["worlds"]:
    raise RuntimeError("no world was analysed — see the table above")


  12/12 — spectral District_D                


### Write

In [4]:
def _frame(v):
    if not v:
        return pd.DataFrame()
    return pd.concat(v, ignore_index=True) if isinstance(v[0], pd.DataFrame) else pd.DataFrame(v)

for name, v in OUT.items():
    _frame(v).to_parquet(OUT_DIR / f"{name}.parquet", index=False)
(OUT_DIR / "manifest.json").write_text(json.dumps({
    "study": STUDY, "version": VERSION, "written": datetime.datetime.now().isoformat(timespec="seconds"),
    "worlds": int(len(WORLDS)), "rhos": RHOS, "tau": TAU, "ratio": RATIO, "issues": issues}, indent=1, default=str))
print({f.stem: len(pd.read_parquet(f)) for f in sorted(OUT_DIR.glob("*.parquet"))})

{'branch': 1980, 'comp_switch': 2040, 'composites': 408, 'groups': 109800, 'hub_elem': 84, 'hub_link': 50652, 'hub_unit_link': 21708, 'hub_units': 36, 'labels': 1812, 'node_district': 1443, 'nodes': 485, 'placement': 192, 'profiles': 15624, 'series_week': 15624, 'stack_gains': 7248, 'switch_a': 28944, 'switch_b': 7236, 'switch_link': 7236, 'topology': 604, 'worlds': 12}
